## Tratamiento de datos 

In [0]:
from pyspark.sql import functions as F

**Tipificacion de tablas **

In [0]:
%sql
USE aerolinea.bronze

In [0]:
%sql
SELECT *FROM aeronaves LIMIT 10

aeronave_id,modelo,fabricante,anio_fabricacion,archivo_origen,fecha_ingesta
A001,B777,Boeing,2013,aeronaves.csv,2026-10-31T03:53:41.992Z
A002,B737,Boeing,2013,aeronaves.csv,2026-10-31T03:53:41.992Z
A003,E190,Embraer,2020,aeronaves.csv,2026-10-31T03:53:41.992Z
A004,B737,Boeing,2013,aeronaves.csv,2026-10-31T03:53:41.992Z
A005,B777,Boeing,2007,aeronaves.csv,2026-10-31T03:53:41.992Z
A006,A350,Airbus,2012,aeronaves.csv,2026-10-31T03:53:41.992Z
A007,A350,Airbus,2015,aeronaves.csv,2026-10-31T03:53:41.992Z
A008,A350,Airbus,2022,aeronaves.csv,2026-10-31T03:53:41.992Z
A009,A320,Airbus,2009,aeronaves.csv,2026-10-31T03:53:41.992Z
A010,B737,Boeing,2012,aeronaves.csv,2026-10-31T03:53:41.992Z


tipificacion de aeronaves 

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW aeronaves_tipificadas AS
SELECT
       UPPER(TRIM(aeronave_id)) AS aeronave_id,
       TRIM(modelo) AS modelo,
       TRIM(fabricante) AS fabricante,
       TRY_CAST(anio_fabricacion AS INT) AS anio_fabricacion,
       archivo_origen
FROM aeronaves

verificacion aeropuerto 

In [0]:
%sql
SELECT * FROM aeronaves_tipificadas limit 5

aeronave_id,modelo,fabricante,anio_fabricacion,archivo_origen
A001,B777,Boeing,2013,aeronaves.csv
A002,B737,Boeing,2013,aeronaves.csv
A003,E190,Embraer,2020,aeronaves.csv
A004,B737,Boeing,2013,aeronaves.csv
A005,B777,Boeing,2007,aeronaves.csv


Tipificacion aeropuertos 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW aeropuertos_tipificados AS
SELECT DISTINCT
       UPPER(TRIM(aeropuerto_id)) AS aeropuerto_id,
       TRIM(nombre) AS nombre, TRIM(ciudad) AS ciudad, TRIM(pais) AS pais,
       TRY_CAST(lat AS DOUBLE) AS lat,
       TRY_CAST(lon AS DOUBLE) AS lon,
       archivo_origen
FROM aerolinea.bronze.aeropuertos

In [0]:
%sql
SELECT * FROM aeropuertos_tipificados limit 5

aeropuerto_id,nombre,ciudad,pais,lat,lon,archivo_origen
MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,40.472,-3.561,aeropuertos.csv
BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,41.297,2.078,aeropuertos.csv
JFK,John F. Kennedy International,Nueva York,EEUU,40.641,-73.778,aeropuertos.csv
LHR,Heathrow,Londres,Reino Unido,51.47,-0.454,aeropuertos.csv
CDG,Charles de Gaulle,París,Francia,49.009,2.547,aeropuertos.csv


**Tipificación de vuelos** 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vuelos_tipificados AS
SELECT DISTINCT
       UPPER(TRIM(vuelo_id)) AS vuelo_id,
       TRY_CAST(fecha AS DATE) AS fecha,
       UPPER(TRIM(origen_id)) AS origen_id,
       UPPER(TRIM(destino_id)) AS destino_id,
       UPPER(TRIM(aeronave_id)) AS aeronave_id,
       LOWER(TRIM(estado)) AS estado,
       TRY_CAST(duracion_min AS INT) AS duracion_min,
       UPPER(TRIM(origen_id)) = UPPER(TRIM(destino_id)) AS ruta_por_revisar,
       fecha_ingesta
FROM vuelos

In [0]:
%sql
SELECT * FROM vuelos_tipificados limit 5

vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min,ruta_por_revisar,fecha_ingesta
V0001,2024-05-21,MAD,JFK,A005,a_tiempo,288,false,2026-10-31T03:53:38.552Z
V0002,2024-05-05,MAD,LHR,A002,cancelado,492,false,2026-10-31T03:53:38.552Z
V0003,2024-05-02,MAD,JFK,A004,a_tiempo,577,false,2026-10-31T03:53:38.552Z
V0004,2024-05-20,MAD,LHR,A004,cancelado,618,false,2026-10-31T03:53:38.552Z
V0005,2024-05-14,BCN,BCN,A010,retrasado,66,true,2026-10-31T03:53:38.552Z


**Tipificación de mantenimiento **

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW mantenimientos_tipificados AS
SELECT DISTINCT
       UPPER(TRIM(mantenimiento_id)) AS mantenimiento_id,
       UPPER(TRIM(aeronave_id)) AS aeronave_id,
       TRY_CAST(fecha AS DATE) AS fecha,
       LOWER(TRIM(tipo)) AS tipo,
       TRY_CAST(costo_usd AS DECIMAL(14,2)) AS costo_usd,
       TRY_CAST(duracion_hr AS DECIMAL(10,2)) AS duracion_hr,
       archivo_origen
FROM mantenimientos

In [0]:
%sql
SELECT * FROM mantenimientos_tipificados limit 5

mantenimiento_id,aeronave_id,fecha,tipo,costo_usd,duracion_hr,archivo_origen
M0001,A001,2024-05-29,inspección,13189.29,16.50,mantenimientos_rds.csv
M0002,A003,2024-05-27,correctivo,36860.78,23.00,mantenimientos_rds.csv
M0003,A010,2024-05-12,correctivo,25147.00,7.70,mantenimientos_rds.csv
M0004,A002,2024-05-23,inspección,44255.92,13.80,mantenimientos_rds.csv
M0005,A007,2024-05-21,correctivo,14791.71,19.10,mantenimientos_rds.csv


##Reglas de contenido

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW control_contenido AS
SELECT 'aeronaves' AS conjunto, COUNT(*) AS errores
FROM aeronaves_tipificadas
WHERE aeronave_id IS NULL OR aeronave_id = '' OR modelo IS NULL OR modelo = ''
   OR fabricante IS NULL OR fabricante = '' OR anio_fabricacion IS NULL
   OR anio_fabricacion < 1900 OR anio_fabricacion > YEAR(current_date())
UNION ALL
SELECT 'aeropuertos', COUNT(*) FROM aeropuertos_tipificados
WHERE aeropuerto_id IS NULL OR aeropuerto_id = '' OR nombre IS NULL OR nombre = ''
   OR ciudad IS NULL OR ciudad = '' OR pais IS NULL OR pais = ''
   OR lat IS NULL OR lon IS NULL OR isnan(lat) OR isnan(lon)
   OR lat NOT BETWEEN -90 AND 90 OR lon NOT BETWEEN -180 AND 180
UNION ALL
SELECT 'vuelos', COUNT(*) FROM vuelos_tipificados
WHERE vuelo_id IS NULL OR vuelo_id = '' OR fecha IS NULL
   OR aeronave_id IS NULL OR aeronave_id = ''
   OR origen_id IS NULL OR origen_id = '' OR destino_id IS NULL OR destino_id = ''
   OR estado IS NULL OR estado NOT IN ('a_tiempo','retrasado','cancelado')
   OR duracion_min < 0
   OR (estado <> 'cancelado' AND (duracion_min IS NULL OR duracion_min = 0))
UNION ALL
SELECT 'mantenimientos', COUNT(*) FROM mantenimientos_tipificados
WHERE mantenimiento_id IS NULL OR mantenimiento_id = '' OR fecha IS NULL
   OR aeronave_id IS NULL OR aeronave_id = '' OR tipo IS NULL
   OR tipo NOT IN ('preventivo','correctivo','inspección')
   OR costo_usd IS NULL OR costo_usd < 0 OR duracion_hr IS NULL OR duracion_hr <= 0

In [0]:
contenido = spark.table("control_contenido")
display(contenido)
assert contenido.filter("errores > 0").count() == 0, "Revisar el contenido antes de publicar Silver."

conjunto,errores
aeronaves,0
aeropuertos,0
vuelos,0
mantenimientos,0


##Revision de conflictos por llave 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW control_llaves AS
SELECT 'aeronaves' AS conjunto, COUNT(*) AS errores FROM (
  SELECT aeronave_id FROM aeronaves_tipificadas GROUP BY aeronave_id HAVING COUNT(*) > 1
) a
UNION ALL
SELECT 'aeropuertos', COUNT(*) FROM (
  SELECT aeropuerto_id FROM aeropuertos_tipificados GROUP BY aeropuerto_id HAVING COUNT(*) > 1
) b
UNION ALL
SELECT 'vuelos', COUNT(*) FROM (
  SELECT vuelo_id FROM vuelos_tipificados GROUP BY vuelo_id HAVING COUNT(*) > 1
) c
UNION ALL
SELECT 'mantenimientos', COUNT(*) FROM (
  SELECT mantenimiento_id FROM mantenimientos_tipificados GROUP BY mantenimiento_id HAVING COUNT(*) > 1
) d

In [0]:
llaves = spark.table("control_llaves")
display(llaves)
assert llaves.filter("errores > 0").count() == 0, "Hay llaves con registros diferentes."

conjunto,errores
aeronaves,0
aeropuertos,0
vuelos,0
mantenimientos,0


Revision de referencias y coherencia temporal

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW control_referencias AS
SELECT 'vuelo_aeronave' AS regla, COUNT(*) AS errores
FROM vuelos_tipificados v LEFT ANTI JOIN aeronaves_tipificadas a ON v.aeronave_id = a.aeronave_id
UNION ALL
SELECT 'vuelo_origen', COUNT(*)
FROM vuelos_tipificados v LEFT ANTI JOIN aeropuertos_tipificados a ON v.origen_id = a.aeropuerto_id
UNION ALL
SELECT 'vuelo_destino', COUNT(*)
FROM vuelos_tipificados v LEFT ANTI JOIN aeropuertos_tipificados a ON v.destino_id = a.aeropuerto_id
UNION ALL
SELECT 'mantenimiento_aeronave', COUNT(*)
FROM mantenimientos_tipificados m LEFT ANTI JOIN aeronaves_tipificadas a ON m.aeronave_id = a.aeronave_id
UNION ALL
SELECT 'fabricacion_posterior_vuelo', COUNT(*)
FROM vuelos_tipificados v JOIN aeronaves_tipificadas a ON v.aeronave_id = a.aeronave_id
WHERE a.anio_fabricacion > YEAR(v.fecha)
UNION ALL
SELECT 'fabricacion_posterior_mantenimiento', COUNT(*)
FROM mantenimientos_tipificados m JOIN aeronaves_tipificadas a ON m.aeronave_id = a.aeronave_id
WHERE a.anio_fabricacion > YEAR(m.fecha)

**Comprobación de las referencias y coherencia**

In [0]:
referencias = spark.table("control_referencias")
display(referencias)
assert referencias.filter("errores > 0").count() == 0, "Revisar las referencias o los años de fabricación."

regla,errores
vuelo_aeronave,0
vuelo_origen,0
vuelo_destino,0
mantenimiento_aeronave,0
fabricacion_posterior_vuelo,0
fabricacion_posterior_mantenimiento,0


**Observaciones visibles**

In [0]:
%sql
SELECT COUNT(*) AS rutas_por_revisar FROM vuelos_tipificados WHERE ruta_por_revisar;
SELECT vuelo_id, fecha, origen_id, destino_id, estado
FROM vuelos_tipificados WHERE ruta_por_revisar ORDER BY vuelo_id;

vuelo_id,fecha,origen_id,destino_id,estado
V0005,2024-05-14,BCN,BCN,retrasado
V0006,2024-05-25,BCN,BCN,retrasado
V0007,2024-05-07,JFK,JFK,retrasado
V0010,2024-05-27,CDG,CDG,a_tiempo
V0017,2024-05-07,CDG,CDG,cancelado
V0022,2024-05-21,BCN,BCN,a_tiempo
V0030,2024-05-08,MAD,MAD,a_tiempo
V0032,2024-05-22,LHR,LHR,retrasado
V0035,2024-05-15,MAD,MAD,a_tiempo
V0038,2024-05-28,JFK,JFK,a_tiempo


**Publiación Capa Silver**

In [0]:
%sql
USE aerolinea.silver

In [0]:
%sql
-- Tabla Silver de aeronaves
CREATE OR REPLACE TABLE aeronaves USING DELTA AS SELECT * FROM aeronaves_tipificadas;

-- Tabla Silver de aeropuertos
CREATE OR REPLACE TABLE aeropuertos USING DELTA AS SELECT * FROM aeropuertos_tipificados;

--Tabla Silver de vuelos
CREATE OR REPLACE TABLE vuelos USING DELTA AS SELECT * FROM vuelos_tipificados;

--Tabla Silver de mantenimientos
CREATE OR REPLACE TABLE mantenimientos USING DELTA AS SELECT * FROM mantenimientos_tipificados;

num_affected_rows,num_inserted_rows


**Comprobación de capas silver**

In [0]:
%sql
select * from aeronaves limit 5

aeronave_id,modelo,fabricante,anio_fabricacion,archivo_origen
A006,A350,Airbus,2012,aeronaves.csv
A008,A350,Airbus,2022,aeronaves.csv
A009,A320,Airbus,2009,aeronaves.csv
A001,B777,Boeing,2013,aeronaves.csv
A003,E190,Embraer,2020,aeronaves.csv


In [0]:
%sql
select * from vuelos limit 5

vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min,ruta_por_revisar,fecha_ingesta
V0009,2024-05-18,MAD,BCN,A002,cancelado,360,false,2026-10-31T03:53:38.552Z
V0014,2024-05-06,CDG,MAD,A003,retrasado,448,false,2026-10-31T03:53:38.552Z
V0015,2024-05-09,CDG,MAD,A006,a_tiempo,294,false,2026-10-31T03:53:38.552Z
V0049,2024-05-10,LHR,LHR,A008,a_tiempo,254,true,2026-10-31T03:53:38.552Z
V0083,2024-05-23,LHR,JFK,A009,cancelado,261,false,2026-10-31T03:53:38.552Z


In [0]:
%sql
select * from aeropuertos limit 5

aeropuerto_id,nombre,ciudad,pais,lat,lon,archivo_origen
MAD,Adolfo Suárez Madrid-Barajas,Madrid,España,40.472,-3.561,aeropuertos.csv
BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España,41.297,2.078,aeropuertos.csv
JFK,John F. Kennedy International,Nueva York,EEUU,40.641,-73.778,aeropuertos.csv
LHR,Heathrow,Londres,Reino Unido,51.47,-0.454,aeropuertos.csv
CDG,Charles de Gaulle,París,Francia,49.009,2.547,aeropuertos.csv


In [0]:
%sql
select * from mantenimientos limit 5

mantenimiento_id,aeronave_id,fecha,tipo,costo_usd,duracion_hr,archivo_origen
M0003,A010,2024-05-12,correctivo,25147.00,7.70,mantenimientos_rds.csv
M0008,A005,2024-05-03,correctivo,7613.00,9.90,mantenimientos_rds.csv
M0010,A006,2024-05-04,preventivo,11258.27,10.80,mantenimientos_rds.csv
M0029,A005,2024-05-08,preventivo,18175.43,18.60,mantenimientos_rds.csv
M0031,A009,2024-05-17,inspección,6991.87,5.80,mantenimientos_rds.csv
